(sequence-search-sort)=

# Searching Ordered and Unordered Data


```{contents}
:local:
:depth: 2
```


An inventory application must distinguish finding an item from ordering the entire catalog. A search contract states which answer you need: the first match, any match, all matches, or a missing result. Representation affects the cost of producing that answer.

Chapter 13 introduced comparison, equality, and operation counts; Chapter 14 introduced array and linked storage. Apply those ideas to a service desk looking for a ticket ID. Every example below owns its array and includes the complete search implementation. Empty input and missing targets are ordinary cases to test, not unusual afterthoughts.


## First-Match Linear Search

```{index} First-Match Linear Search
```


Linear search visits values in order and returns immediately when equality succeeds. It accepts unsorted data. Return the first matching index or -1; preserve the input and do not treat index zero as a missing result.

The processed prefix contains no match. When the loop reaches the end, every position has been checked, so -1 is justified. A match at index k needs k+1 equality checks. A missing target needs n checks; an empty array needs zero. This is O(n) worst-case work under the fixed-cost comparison model.


In [1]:
using System;
using System.Collections.Generic;

int[] ids = {42, 17, 42, 8};
Console.WriteLine(BasicSearch.LinearFirst(ids, 42, EqualityComparer<int>.Default));
Console.WriteLine(BasicSearch.LinearFirst(ids, 99, EqualityComparer<int>.Default));
Console.WriteLine(BasicSearch.LinearFirst(Array.Empty<int>(), 42, EqualityComparer<int>.Default));

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}


0
-1
-1


The results are 0, -1, and -1. An equality comparer makes the matching rule explicit. For symbolic IDs you might choose case-sensitive or case-insensitive string equality, but use the same policy consistently. Matching a customer's name is not necessarily the same as identifying the customer's record.

For linked nodes, linear traversal is also natural: follow each Next reference once. Repeatedly requesting an item by numeric index can rescan the list from its head and accidentally turn a linear task into quadratic work.


## Binary Search Requires the Right Order

```{index} Binary Search Requires the Right Order
```


(binarysearching)=

Binary search repeatedly discards half of a sorted, random-access sequence. The data must be sorted under the same ordering comparer used by the search. Sorting by price and then searching by product ID does not meet that precondition.

The interval [low, high) contains all remaining candidates. Start low at zero and high at Length. Compare the middle value: equality returns its index; a smaller middle excludes the left portion and the middle; a larger middle excludes the middle and the right portion. The interval shrinks on every unsuccessful comparison.

The safe midpoint expression low + (high - low) / 2 avoids adding two large indexes. High is a boundary, not always a valid element index. The method never indexes an empty interval.


In [1]:
using System;
using System.Collections.Generic;

int[] ids = {8, 17, 42, 42, 60};
Console.WriteLine(BasicSearch.BinaryAny(ids, 17, Comparer<int>.Default));
Console.WriteLine(BasicSearch.BinaryAny(ids, 99, Comparer<int>.Default));
Console.WriteLine(BasicSearch.BinaryAny(Array.Empty<int>(), 17, Comparer<int>.Default));

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}


1
-1
-1


There are at most floor(log2 n)+1 comparisons for a nonempty array, and O(1) auxiliary storage in this iterative version. The method returns any matching index: duplicate values do not imply the first matching index or the first arrival.

Trace a missing target 20 in [8,17,42,42,60]. The interval narrows until its boundaries meet between 17 and 42. Missing searches and singleton ranges are essential tests because an incorrect boundary update can skip candidates or fail to make progress.


## Library Results Have Their Own Contract

```{index} Library Results Have Their Own Contract
```


Array.IndexOf uses -1 for a missing value. Array.BinarySearch returns a negative encoded insertion position instead; test result >= 0 before using it as an index. The bitwise complement ~result decodes the position where a missing value could be inserted while preserving order.

Do not mix a teaching method's sentinel with a library method's contract. Read the API promise and check boundaries before using a returned index. The required learning is the sign test; insertion-position encoding is an applied extension.


In [1]:
using System;
using System.Collections.Generic;

int[] ids = {8, 17, 42};
int found = Array.BinarySearch(ids, 20);
Console.WriteLine(found >= 0 ? "Found" : "Missing");
if (found < 0) Console.WriteLine($"Insertion position: {~found}");


Missing
Insertion position: 2


## Choose a Search for the Workload

```{index} Choose a Search for the Workload
```


One scan may be cheaper than sorting solely to answer one query. Repeated searches on static data can justify a sorted snapshot: sorting once has its own preparation cost, followed by logarithmic searches. Updates may invalidate that snapshot. A dictionary supports exact-key access but does not replace an ordered range query.

State the input size, required answer, update pattern, equality policy, and representation before choosing an algorithm. A stopwatch observation can support a particular measurement; it cannot establish a general growth bound. The next sections make sorting work visible and connect preparation to a complete reporting workload.


## Practice: State the Matching Policy

```{index} Practice: State the Matching Policy
```


Find the first case-insensitive ticket ID without sorting or changing the input.


In [ ]:
using System;
using System.Collections.Generic;

string[] ids = {"T-100", "t-100", "T-200"};
int first = BasicSearch.LinearFirst(ids, "t-100", StringComparer.Ordinal /* TODO: choose case-insensitive equality */);
Console.WriteLine(first);

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}


In [1]:
using System;
using System.Collections.Generic;

string[] ids = {"T-100", "t-100", "T-200"};
int first = BasicSearch.LinearFirst(ids, "t-100", StringComparer.OrdinalIgnoreCase);
Console.WriteLine(first);

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}


0


```{rubric} Footnotes
```
[^1]: Binary search needs compatible ordering and random access. More detailed duplicate-boundary searches and algorithm proofs remain in optional Part V.
